# Day 16: Parallel Agents — Run Two at Once

**30-Day AI Agent Challenge: OpenAI Agents SDK vs LangGraph vs CrewAI**

---

## Today's Concept

> Why do one thing at a time when you can do two simultaneously?

Parallel execution is what makes multi-agent systems powerful. Instead of research
THEN write sequentially, you research AND write at the same time, then merge.

## What You'll Build
- Two research agents running in parallel
- A merge step that combines their findings
- Parallel execution in all 3 frameworks

In [8]:
# ── Setup ───────────────────────────────────────────────
import sys
sys.path.insert(0, "..")
from shared import check_and_report, print_config, disable_openai_tracing, get_openai_agents_model, get_langgraph_model, get_crewai_llm
check_and_report()
disable_openai_tracing()
print("=" * 50)
print("DAY 16 SETUP COMPLETE")
print("=" * 50)
print_config()

Python: 3.12.13

All required packages installed.
Optional (missing): langchain-openai, langgraph-checkpoint-memory, ipywidgets

Ollama: connected (10 model(s) available)

DAY 16 SETUP COMPLETE
  Provider: OLLAMA
  Model:    qwen2.5:3b
  Ollama:   http://localhost:11434


## Today's Architecture

```
       ┌──────────────────────────┐
       │ Each branch has its own  │
       │ prompt (no shared input) │
       └────────────┬─────────────┘
                    │ fan-out
         ┌──────────┴──────────┐
         │                     │
     ┌───▼──────┐        ┌─────▼────┐
     │   Tech   │        │ Business │
     │ Research │        │ Research │
     └────┬─────┘        └─────┬────┘
          └─────────┬──────────┘
                    │ fan-in
             ┌──────▼──────┐
             │ Synthesize  │
             └─────────────┘
```

Two researchers explore different angles in parallel, then a synthesizer combines their findings.
**Note:** each branch has its own prompt baked into the agent — there is no shared
"user input" node at the top. The fan-out is conceptual; each framework wires its
own per-branch prompt (see the cells below).


---
## Framework 1: OpenAI Agents SDK — asyncio.gather

In [9]:
from agents import Agent, Runner
import asyncio

model = get_openai_agents_model()

tech_researcher = Agent(
    name="Tech Researcher",
    instructions="Research the TECHNICAL aspects of AI agents: architecture, frameworks, implementation patterns.",
    model=model,
)

biz_researcher = Agent(
    name="Business Researcher",
    instructions="Research the BUSINESS aspects of AI agents: ROI, use cases, market adoption, cost savings.",
    model=model,
)

synthesizer = Agent(
    name="Synthesizer",
    instructions="Combine research findings into a coherent summary covering both technical and business angles.",
    model=model,
)

print("=" * 60)
print("OPENAI AGENTS SDK — PARALLEL RESEARCH (asyncio.gather)")
print("=" * 60)

# Run both researchers IN PARALLEL — gather kicks them off concurrently
tech_result, biz_result = await asyncio.gather(
    Runner.run(tech_researcher, "What are the key technical aspects of AI agents?"),
    Runner.run(biz_researcher, "What are the key business benefits of AI agents?"),
)

print("\n[TECH]", tech_result.final_output)
print("\n[BIZ]", biz_result.final_output)

# Synthesize
combined = f"TECHNICAL: {tech_result.final_output}\n\nBUSINESS: {biz_result.final_output}"
final = await Runner.run(synthesizer, f"Synthesize these research findings:\n\n{combined}")
print(f"\n[SYNTHESIS] {final.final_output}")


OPENAI AGENTS SDK — PARALLEL RESEARCH (asyncio.gather)

[TECH] The key technical aspects of AI agents are numerous and interrelated, encompassing both their foundational techniques and practical applications. Here’s a breakdown focusing on architectural design, tools/frameworks used to implement them, and common implementation patterns:

### 1. **Architecture**
AI agent architectures can be broadly categorized into four main groups:
- **Supervised Learning:** Where agents are trained through examples. The training data is labeled in some way, guiding the model's learning.
- **Unsupervised Learning:** Here, AI models analyze unlabelled data to recognize patterns, structure, and categories within it. Unlike supervised learning where we directly provide a correct answer for each input-output pair.
- **Reinforcement Learning (RL):** Agents learn by trial and error in an environment with rewards or punishments, aiming to reach a goal state. The agent's performance is continuously evaluated 

---
## Framework 2: LangGraph — Parallel Branches

In [7]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from typing_extensions import TypedDict, Annotated
import operator

model = get_langgraph_model()

class ParallelState(TypedDict):
    messages: Annotated[list, operator.add]
    tech_research: str
    biz_research: str

def tech_node(state: ParallelState) -> dict:
    r = model.invoke([
        SystemMessage(content="Research technical aspects of AI agents."),
        HumanMessage(content="What are the key technical aspects of AI agents?"),
    ])
    return {"tech_research": r.content}

def biz_node(state: ParallelState) -> dict:
    r = model.invoke([
        SystemMessage(content="Research business benefits of AI agents."),
        HumanMessage(content="What are the key business benefits of AI agents?"),
    ])
    return {"biz_research": r.content}

def synthesize_node(state: ParallelState) -> dict:
    combined = f"TECHNICAL: {state['tech_research']}\nBUSINESS: {state['biz_research']}"
    r = model.invoke([
        SystemMessage(content="Synthesize into a coherent summary."),
        HumanMessage(content=combined),
    ])
    return {"messages": [r]}

builder = StateGraph(ParallelState)
builder.add_node("tech", tech_node)
builder.add_node("biz", biz_node)
builder.add_node("synthesize", synthesize_node)
# START splits into two parallel branches
builder.add_edge(START, "tech")
builder.add_edge(START, "biz")
# Both branches merge into synthesize
builder.add_edge("tech", "synthesize")
builder.add_edge("biz", "synthesize")
builder.add_edge("synthesize", END)

graph = builder.compile()

print("=" * 60)
print("LANGGRAPH — PARALLEL BRANCHES")
print("=" * 60)

result = graph.invoke({"messages": [HumanMessage(content="Go")], "tech_research": "", "biz_research": ""})
print(f"\n[TECH] {result['tech_research']}")
print(f"\n[BIZ] {result['biz_research']}")
print(f"\n[SYNTHESIS] {result['messages'][-1].content}")

LANGGRAPH — PARALLEL BRANCHES

[TECH] The key technical aspects of AI agents encompass several areas that enable them to perceive their environment, understand their surroundings, make decisions based on this understanding, and take actions accordingly. Here are some of the major technical aspects:

1. **Representation Learning**: This involves how an AI agent represents information about its environment and goals internally. Techniques in representation learning include feature engineering (for simpler agents) or more complex approaches like deep neural networks for capturing intricate relationships between features.

2. **Planning**: For tasks that require long-term goals, planning is crucial. Agents need to predict the outcomes of their actions over time and choose strategies based on these predictions. Planning can be deterministic or probabilistic depending on the nature of the environment and the level of uncertainty involved.

3. **Reinforcement Learning (RL)**: This involves ag

---
## Framework 3: CrewAI — Parallel via Flows

Real fan-out / fan-in using `@start()` + `@listen()` + `and_()`:
`kick_off` fans out to `tech_branch` and `biz_branch` (run concurrently),
then `synthesize` waits for **both** via `and_(tech_branch, biz_branch)`.


In [10]:
from crewai import Agent, Task, Crew, Process
from crewai.flow.flow import Flow, listen, start, and_

llm = get_crewai_llm()

tech = Agent(role="Tech Researcher", goal="Research technical AI agent aspects", backstory="Technical expert.", llm=llm)
biz = Agent(role="Business Researcher", goal="Research business AI agent benefits", backstory="Business analyst.", llm=llm)
writer = Agent(role="Synthesizer", goal="Combine research into a summary", backstory="You merge perspectives.", llm=llm)

class ParallelResearchFlow(Flow):
    @start()
    def kick_off(self):
        # ponytail: dict state, no pydantic model — this flow has 3 string fields, that's it
        self.state["tech"] = ""
        self.state["biz"] = ""

    @listen(kick_off)
    def tech_branch(self):
        task = Task(description="Research 3 key technical aspects of AI agents.", expected_output="3 technical findings.", agent=tech)
        r = Crew(agents=[tech], tasks=[task], process=Process.sequential, verbose=False).kickoff()
        self.state["tech"] = str(r)

    @listen(kick_off)
    def biz_branch(self):
        task = Task(description="Research 3 key business benefits of AI agents.", expected_output="3 business findings.", agent=biz)
        r = Crew(agents=[biz], tasks=[task], process=Process.sequential, verbose=False).kickoff()
        self.state["biz"] = str(r)

    # and_() waits for BOTH branches — true fan-out / fan-in
    @listen(and_(tech_branch, biz_branch))
    def synthesize(self):
        combined = f"TECHNICAL:\n{self.state['tech']}\n\nBUSINESS:\n{self.state['biz']}"
        task = Task(description=f"Combine into a coherent summary.\n\n{combined}", expected_output="Combined summary.", agent=writer)
        r = Crew(agents=[writer], tasks=[task], process=Process.sequential, verbose=False).kickoff()
        self.state["final"] = str(r)

print("=" * 60)
print("CREWAI — PARALLEL VIA FLOWS (tech_branch || biz_branch -> synthesize)")
print("=" * 60)

flow = ParallelResearchFlow()
await flow.kickoff_async()
print(flow.state.get("final", "<no final output>"))


CREWAI — PARALLEL VIA FLOWS (tech_branch || biz_branch -> synthesize)


╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: ParallelResearchFlow                                                                                     │
│  ID: d1d8db0b-e5ed-4554-a887-a72acd444dee                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: ParallelResearchFlow                                                                                     │
│  ID: d1d8db0b-e5ed-4554-a887-a72acd444dee                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Flow started with ID: d1d8db0b-e5ed-4554-a887-a72acd444dee

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: ParallelResearchFlow                                                                                     │
│  ID: d1d8db0b-e5ed-4554-a887-a72acd444dee                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Sure, here's a combined technical and business perspective into a coherent summary:

**Combined Summary:**

Enhancing Intelligent Agent Solutions with AI in Business

This summary synthesizes technical enhancements and their corresponding benefits within businesses by examining how intelligent agents using Artificial Intelligence (AI) are improving efficiency, service delivery, and forecasting capabilities.

**Enhanced Customer Service through Intelligent Agents**

1. **Technical Aspect**: AI agents leverage Natural Language Processing (NLP), enabling them to understand customer inquiries accurately and respond effectively. They employ various NLP techniques including text processing tools and advanced analysis methods for sentiment analysis, question answering, and more.
2. **Business Benefit**: These same AI capabilities enable intelligent agent systems to handle a wide range of customer queries quickly, enhancing satisfaction by providing immediate responses while freeing human staf

---
## Comparison: Parallel Execution

| Framework | What this notebook does | True parallelism? | Notes |
|---|---|---|---|
| OpenAI SDK | `asyncio.gather()` over two `Runner.run` calls | **Yes** | Async required; both researchers run concurrently |
| LangGraph | Two edges from `START` (`tech`, `biz`) | **Yes** | Automatic — graph topology forks for you |
| CrewAI | Flow with `@start()` + `and_()` fan-out / fan-in | **Yes** | Two `@listen(kick_off)` branches run concurrently; `and_()` joins them |

**Key insight:** All three frameworks now run genuinely in parallel.
LangGraph is the most natural — the graph structure makes parallel branches explicit
and the merge point visible. OpenAI SDK makes you manage async yourself via `asyncio.gather`.
CrewAI sits in between: Flows give you declarative fan-out via `@listen()` + `and_()`,
but each branch still wraps a `Crew` so there's more wiring than LangGraph.


---
## Timing: Sequential vs Parallel — Measured

The post below claims parallel beats sequential. Let's measure it.
Same two researchers, same model — the only difference is `asyncio.gather` vs two sequential `await`s.
(Using OpenAI SDK because it has the cleanest async API; the same wall-clock win applies to LangGraph's
graph fork and CrewAI's Flow fan-out vs their sequential counterparts.)


In [11]:
import time, asyncio
from agents import Agent, Runner

model = get_openai_agents_model()
t_agent = Agent(name="Tech", instructions="List 3 technical AI agent concepts.", model=model)
b_agent = Agent(name="Biz",  instructions="List 3 business AI agent benefits.",  model=model)

# Sequential — one, then the other
t0 = time.perf_counter()
await Runner.run(t_agent, "Go")
await Runner.run(b_agent, "Go")
seq_s = time.perf_counter() - t0

# Parallel — both at once via asyncio.gather
t0 = time.perf_counter()
await asyncio.gather(
    Runner.run(t_agent, "Go"),
    Runner.run(b_agent, "Go"),
)
par_s = time.perf_counter() - t0

print(f"Sequential: {seq_s:.2f}s")
print(f"Parallel:   {par_s:.2f}s")
print(f"Speedup:    {seq_s/par_s:.2f}x  (saved {seq_s - par_s:.2f}s)")


Sequential: 25.23s
Parallel:   17.33s
Speedup:    1.46x  (saved 7.90s)


## Key Takeaway

Parallel execution makes multi-agent systems practical. LangGraph's graph structure
makes it the most natural — two edges from START create automatic parallelism,
and the merge point is explicit in the graph. The timing cell above shows the
speedup is measurable, not theoretical.

---

